# IKEA_ASM FULL — 254 train / 117 test trên Colab T4

Chọn kernel Colab T4 và **Run All**. Notebook mặc định full split, không lấy subset và không cần sửa cấu hình. Xác nhận quyền Google Drive nếu được yêu cầu. Source đã nhúng; dữ liệu giữ trên Drive.

Luồng: kiểm đủ 371 video dev3 → trích I3D từng video → train ProTAS 50 epoch (12700 optimizer steps) → đánh giá toàn bộ 117 test. Features của test chỉ là đầu vào encoder đóng băng; graph và supervision train chỉ dùng 254 train. Test chỉ chấm sau fixed-final checkpoint.

Features/checkpoint/log lưu Drive. Nếu phiên bị ngắt: kết nối lại T4 rồi Run All cùng notebook/cấu hình; recording features đã hoàn tất được bỏ qua và train tiếp checkpoint. Video đang trích dở phải làm lại; phần train sau checkpoint cuối có thể phải làm lại. Không bảo đảm hoàn thành full trong một phiên miễn phí.

Đây là ProTAS thích nghi sang IKEA, frozen RGB I3D 1024 chiều so với 2048 trong paper, không tuyên bố tái lập điểm SOTA. Clone graph reference để CPU checkpoint load đúng. Run full được tách riêng các pilot trước.


## 1. Tự mount Drive
Cell tự dùng Drive đã mount, hoặc mở bước cấp quyền của Google. Đây là thao tác xác nhận tài khoản bắt buộc khi Google yêu cầu; không cần bấm Mount Drive riêng trước Run All.


In [ ]:
from pathlib import Path
# Reset only this pipeline's phase flags. Failed cells cannot reuse stale
# feature_report/checkpoints from a previous Run All in the same kernel.
_IKEA_PHASE = dict(drive=False, source=False, environment=False, config=False,
                   data=False, features=False, trained=False)
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
_IKEA_PHASE['drive'] = Path('/content/drive/MyDrive').is_dir()
if not _IKEA_PHASE['drive']:
    raise RuntimeError('Google Drive chưa được cấp quyền/mount. Xác nhận tài khoản Google để tiếp tục.')
print('Drive ready:', _IKEA_PHASE['drive'])


## 2. Tự nạp source được nhúng trong notebook
Payload chỉ chứa các file runtime trong allowlist, không có video, weights, notebook khác hoặc credentials. Kiểm SHA256 và từng file trước khi nạp vào thư mục có phiên bản dưới `/content`; không cần module `experiences` tồn tại từ trước.


In [ ]:
# Self-contained source: no upload, repository URL, or project import needed.
from pathlib import Path, PurePosixPath
import base64, hashlib, importlib, json, os, signal, stat, subprocess, sys, zipfile
if not isinstance(globals().get('_IKEA_PHASE'), dict):
    _IKEA_PHASE = {}
_IKEA_PHASE['source'] = False
SOURCE_SHA256 = 'a1bf19f16f47228aadbb2e62e58d075ae587ef52baa238251f73094d888c768b'
EMBEDDED_SOURCE = (
    '...'
    'c29uUEsFBgAAAAAOAA4AKAQAAG1pAAAAAA=='
)
def unpack_bundle(archive_path, destination):
    """Verify all entries before extracting into a new, versioned code directory."""
    destination = Path(destination)
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        if len(entries) > 500 or sum(i.file_size for i in entries) > 32 * 1024 * 1024:
            raise ValueError("Source bundle exceeds the project limits")
        manifest = json.loads(archive.read("source_bundle_manifest.json"))
        names = {i.filename for i in entries}
        if (manifest.get("purpose") != "code_only_no_datasets_no_credentials"
                or len(names) != len(entries)
                or names != set(manifest["files"]) | {"source_bundle_manifest.json"}):
            raise ValueError("Source bundle manifest does not match archive")
        for entry in entries:
            path = PurePosixPath(entry.filename)
            if (path.is_absolute() or ".." in path.parts or "\\" in entry.filename
                    or stat.S_ISLNK(entry.external_attr >> 16)):
                raise ValueError("Unsafe source bundle path")
            if entry.filename in manifest["files"]:
                if hashlib.sha256(archive.read(entry)).hexdigest() != manifest["files"][entry.filename]:
                    raise ValueError("Source checksum mismatch: " + entry.filename)
        if destination.exists():
            # Reuse only if the complete expected source is still identical.
            if any(not (destination / name).is_file()
                   or hashlib.sha256((destination / name).read_bytes()).hexdigest() != digest
                   for name, digest in manifest["files"].items()):
                raise ValueError("Existing code differs; use a new source bundle")
        else:
            destination.mkdir(parents=True)
            archive.extractall(destination)
    return destination

_payload = base64.b64decode(EMBEDDED_SOURCE, validate=True)
if hashlib.sha256(_payload).hexdigest() != SOURCE_SHA256:
    raise RuntimeError('Embedded source checksum mismatch; open the unmodified notebook.')
# CONTENT_ROOT is overridable only for isolated local verification.
_content = Path(globals().get('CONTENT_ROOT', '/content'))
_content.mkdir(parents=True, exist_ok=True)
_archive = _content / ('ikea_source_' + SOURCE_SHA256[:12] + '.zip')
_archive.write_bytes(_payload)
CODE_DIR = _content / ('VideoSegmentationIKEAProTAS_' + SOURCE_SHA256[:12])
unpack_bundle(_archive, CODE_DIR)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
# Later project commands use isolated child interpreters from this directory,
# so an older project imported in this kernel cannot shadow the embedded code.
print('Source ready:', CODE_DIR)
print('Source SHA256:', SOURCE_SHA256)
_IKEA_PHASE['source'] = True
del _payload


## 3. Thư viện và kiểm tra GPU
Code ProTAS tác giả dùng FP32; notebook không tự đổi sang AMP hoặc giảm architecture. Nếu chưa có GPU thì dừng trước khi tải/trích video.


In [ ]:
_IKEA_PHASE['environment'] = False
if not _IKEA_PHASE.get('drive') or not _IKEA_PHASE.get('source'):
    raise RuntimeError('Bước Drive/source chưa hoàn tất. Chạy Run All từ đầu và xác nhận quyền Google.')
if 'CODE_DIR' not in globals() or not (CODE_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('Chạy Run All từ đầu để nạp embedded source trước.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(CODE_DIR) + '[train,video]'], check=True)
import torch
def check_t4():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU assigned. Request a Colab T4 first.")
    name = torch.cuda.get_device_name(0)
    if "T4" not in name.upper():
        raise RuntimeError(f"Requested T4 but assigned {name}; inspect the session before training.")
    # Verify allocation AND an actual CUDA operation, without changing a model.
    with torch.no_grad():
        x = torch.ones((16, 16), device="cuda")
        value = float((x @ x).sum().item())
    if value != 4096.0:
        raise RuntimeError("CUDA preflight failed")
    return {"gpu": name, "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
            "torch": torch.__version__, "cuda": torch.version.cuda,
            "cuda_operation_passed": True, "optimizer_steps": 0}

print(check_t4())
torch.set_num_threads(2)
_IKEA_PHASE['environment'] = True


## 4. Cấu hình FULL và tìm dataset
Giữ `SCOPE='full'`, 50 epoch, camera dev3, split gốc 254/117. Dataset phải đã giải nén đầy đủ trong Drive. Notebook từ chối thiếu video thay vì tự giảm về pilot.


In [ ]:
_IKEA_PHASE['config'] = False
if not _IKEA_PHASE.get('environment'):
    raise RuntimeError('Môi trường/GPU chưa sẵn sàng; xử lý cell lỗi trước khi tiếp tục.')
_discover = 'from experiences.ikea_protas.bootstrap import locate_dataset; print(locate_dataset("/content/drive/MyDrive"))'
IKEA_ROOT = subprocess.check_output([sys.executable, '-c', _discover], text=True).strip()
print('IKEA found:', IKEA_ROOT)
OUTPUT_ROOT = '/content/drive/MyDrive/VideoSegmentationRuns/IKEA_ProTAS'
SCOPE = 'full'
RUN_TAG = 'full_v2_50epochs'
STRIDE_FRAMES = 2  # every 2 native frames; ~15 Hz only for a 30 FPS source
TOTAL_EPOCHS = 50
MINIMUM_STEPS = 0
EPOCHS_THIS_SESSION = 50
MAX_NEW_VIDEOS = 371
TRAIN_SESSION_SECONDS = 5400

if SCOPE != 'full' or not RUN_TAG or Path(RUN_TAG).name != RUN_TAG or RUN_TAG in {'.', '..'}:
    raise ValueError('Invalid scope/run tag')
if any(type(n) is not int or n < 1 for n in (STRIDE_FRAMES, TOTAL_EPOCHS, EPOCHS_THIS_SESSION, MAX_NEW_VIDEOS)):
    raise ValueError('Budgets/stride/epochs phải là số nguyên dương')
PARENT = Path(OUTPUT_ROOT)
RUN_DIR = PARENT / SCOPE / f'stride{STRIDE_FRAMES}' / RUN_TAG
CACHE_DIR = PARENT / f'i3d_rgb_cache_stride{STRIDE_FRAMES}'
DATA_DIR = RUN_DIR / 'prepared'
WEIGHTS = PARENT / 'assets/rgb_imagenet.pt'
INDEX_PATH = RUN_DIR / 'index.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset:', IKEA_ROOT, '| scope:', SCOPE, '| run:', RUN_DIR)

_IKEA_PHASE['config'] = True


## 5. Lấy đúng code tác giả và dựng index từ nhãn gốc
Setup clone hai repo chính thức ở commit đã pin và kiểm SHA256 trọng số I3D Kinetics/ImageNet. Index đọc SQLite + `indexing_files`, camera `dev3`, split cross-environment; không cần điền manifest goal-level bằng tay. Full sẽ báo lỗi nếu thiếu video, không âm thầm coi subset là full.


In [ ]:
_IKEA_PHASE['data'] = False
if not _IKEA_PHASE.get('config'):
    raise RuntimeError('Bước trước chưa hoàn tất; xử lý cell lỗi hoặc chạy lại Run All.')
# AUTO_PHASE_BODY
subprocess.run([sys.executable, '-m', 'experiences.ikea_protas', 'setup', '--weights', str(WEIGHTS)], check=True)
subprocess.run([sys.executable, '-m', 'experiences.ikea_protas', 'index',
                '--dataset-root', IKEA_ROOT, '--scope', SCOPE, '--output', str(INDEX_PATH)], check=True)
index = json.loads(INDEX_PATH.read_text())
print('Classes:', len(index['labels']), '| indexed:', len(index['records']), '| official split:', index['original_split_counts'])
for r in index['records'][:8]:
    print(r['split'], r['nframes'], r['recording_id'])

# FULL_SPLIT_GUARD
actual_counts = {split: sum(r['split'] == split for r in index['records']) for split in ('train', 'test')}
if index['scope'] != 'full' or actual_counts != {'train': 254, 'test': 117}:
    raise RuntimeError('FULL cần đúng 254 train /117 test; thực tế: ' + str(actual_counts))
print('FULL verified:', actual_counts, '| total videos:', len(index['records']))

_IKEA_PHASE['data'] = True


## 6. Trích features đủ 371 video và lưu Drive
Output `Extracting: ...` và `decoded=...` thể hiện tiến độ từng video. Lượt đầu có thể mất nhiều phiên vì phải đọc/decode/I3D toàn dataset. Video gốc chỉ copy local từng file, không tải cả dataset vào /content. Chỉ khi đủ 371 cache hoàn tất mới train; Run All sau reconnect dùng lại cache đã lưu.


In [ ]:
_IKEA_PHASE['features'] = False
if not _IKEA_PHASE.get('data'):
    raise RuntimeError('Bước trước chưa hoàn tất; xử lý cell lỗi hoặc chạy lại Run All.')
# AUTO_PHASE_BODY
EXTRACTION_REPORT = RUN_DIR / 'extraction_status.json'
# Dedicated child releases encoder memory on exit and bounds glibc arenas.
subprocess.run([sys.executable, '-u', '-m', 'experiences.ikea_protas', 'extract',
                '--index', str(INDEX_PATH), '--cache-dir', str(CACHE_DIR), '--weights', str(WEIGHTS),
                '--stride', str(STRIDE_FRAMES), '--batch-size', '1', '--device', 'cuda',
                '--max-new-records', str(MAX_NEW_VIDEOS), '--scratch', '/content/ikea_video_scratch',
                '--report', str(EXTRACTION_REPORT)], check=True,
               env={**os.environ, 'MALLOC_ARENA_MAX': '2', 'OMP_NUM_THREADS': '2'})
feature_report = json.loads(EXTRACTION_REPORT.read_text())
print(json.dumps(feature_report, indent=2))
if not feature_report['all_ready']:
    print('Chưa đủ features. Chạy lại cell 6 ở đợt sau; các cell train/test bên dưới sẽ bỏ qua.')

_IKEA_PHASE['features'] = bool(feature_report['all_ready'])


## 7. Train đủ 254 video, 50 epoch
Output `epoch_in_progress`, `recording`/`train_recordings=254`, `loss`, `optimizer_steps` cho biết tiến độ. Lịch đủ là 12700 updates. Mỗi 50 recording và cuối epoch lưu checkpoint; budget train của mỗi lượt là 5400 giây, kiểm tra sau recording. Chu kỳ 50 recording là thiết lập resume của runner, không phải lịch 5 epoch của tác giả. Khi phiên bị thu hồi đột ngột có thể phải làm lại gần 50 recording sau checkpoint cuối. Khi cell dừng theo budget cũng lưu checkpoint; Run All tiếp tục và vẫn giữ tổng 50 epoch.


In [ ]:
def run_logged(command, run_dir):
    """Stream to the client and Drive; interruption stops the child as well."""
    run_dir = Path(run_dir).resolve()
    drive = Path("/content/drive/MyDrive").resolve()
    if not drive.is_dir() or not run_dir.is_relative_to(drive):
        raise ValueError("Place output_dir under mounted /content/drive/MyDrive for durable checkpoints")
    run_dir.mkdir(parents=True, exist_ok=True)
    with (run_dir / "console.log").open("a", buffering=1) as log:
        log.write("\nCOMMAND " + json.dumps(command) + "\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            code = process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
        if code:
            raise subprocess.CalledProcessError(code, command)

_IKEA_PHASE['trained'] = False
if not _IKEA_PHASE.get('data'):
    raise RuntimeError('Index/dữ liệu chưa sẵn sàng; không dùng trạng thái cũ để train.')
if _IKEA_PHASE.get('features'):
    subprocess.run([sys.executable, '-m', 'experiences.ikea_protas', 'prepare',
                    '--index', str(INDEX_PATH), '--cache-dir', str(CACHE_DIR), '--output-dir', str(DATA_DIR)], check=True)
    run_logged([sys.executable, '-u', '-m', 'experiences.ikea_protas', 'fit',
                '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR / 'model'),
                '--epochs', str(TOTAL_EPOCHS), '--minimum-steps', str(MINIMUM_STEPS), '--graph-reference-fix', '--epochs-this-session', str(EPOCHS_THIS_SESSION),
                '--checkpoint-every', '50', '--max-seconds', str(TRAIN_SESSION_SECONDS),
                '--local-dir', '/content/ikea_protas_features', '--device', 'cuda'], RUN_DIR)
    _IKEA_PHASE['trained'] = True
else:
    print('Train chưa chạy: cần hoàn tất features; Run All lần sau sẽ tiếp tục cache.')


## 8. Đánh giá đủ 117 test khi hoàn tất 50 epoch
Ghi Accuracy, Accuracy bỏ NA, Edit, F1@10/25/50, prediction theo frame và segment `[start,end)`. Chưa đủ 50 epoch thì chỉ báo tiến độ, chưa chạy test.


In [ ]:
if not _IKEA_PHASE.get('trained'):
    print('Chưa đánh giá: pipeline train/features của đợt này chưa hoàn tất.')
else:
    # AUTO_EVALUATION_BODY
    last = RUN_DIR / 'model/last.pt'
    if last.is_file():
        saved = torch.load(last, map_location='cpu', weights_only=True)
        print('Completed epochs:', saved['completed_epochs'], '/', saved['config']['total_epochs'],
              '| optimizer steps:', saved['optimizer_steps'], '| next recording:', saved['next_recording'])
        if saved['completed_epochs'] == saved['config']['total_epochs'] and not saved['partial']['recordings']:
            run_logged([sys.executable, '-u', '-m', 'experiences.ikea_protas', 'evaluate',
                        '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR / 'model'),
                        '--local-dir', '/content/ikea_protas_features', '--device', 'cuda'], RUN_DIR)
            result = json.loads((RUN_DIR / 'model/test.json').read_text())
            print(json.dumps({'scope': result['scope'], 'metrics': result['metrics'],
                              'test_recordings': len(result['records'])}, indent=2))
        else:
            print('Checkpoint đã lưu. Chạy lại cell 7 để tiếp tục, chưa dùng test để chọn model.')
    else:
        print('Chưa có checkpoint; xem trạng thái features ở cell 6.')


## Tài liệu và phạm vi
- Notebook là một file độc lập; bước setup tự lấy [ProTAS](https://github.com/Yuhan-Shen/ProTAS) và [IKEA toolkit](https://github.com/IkeaASM/IKEA_ASM_Dataset) tại commit cố định.
- `pilot` là kiểm tra đầu-cuối trên ít video, không đại diện full dataset; JSON ghi scope. Không dùng test để chọn checkpoint.
- Run All sau khi reconnect dùng lại features/checkpoint đã commit trên Drive. Không có keep-alive hoặc bảo đảm GPU miễn phí luôn khả dụng.
- Khi sửa runtime source trong repo: chạy `python3 -m experiences.ikea_protas.build_notebook` để cập nhật payload. Không cần đóng gói/upload ZIP.
